In [1]:
import pandas as pd
import numpy as np

# Wczytanie trójkąta z Excela
df = pd.read_excel(
    r"D:\MOdel_wewnetrzny\test_pz\trojkat_szkodowy_22x22.xlsx",
    sheet_name=0,
    index_col=0
)

# Konwersja do NumPy
triangle = df.to_numpy(dtype=np.float64)


In [2]:
wagi = pd.read_excel(
    "wagi.xlsx",
    index_col=0
).to_numpy(dtype=np.int64)


In [3]:
from mack import kalibracja_macka
L, f, fr, sigma2, nf, ns = kalibracja_macka(
    triangle,
    wagi
)

In [5]:
# Wagi regresji sigma
v_sigma = np.maximum(ns - 1, 0).astype(np.float64)
from mack import dopasuj_sigma

sigma2_fit, b0, b1, wynik = dopasuj_sigma(
    sigma2=sigma2,
    curve_type="weibull",
    fit_from=5,
    fit_to=14,
    smooth_from=5,
    smooth_to=21,
    v=v_sigma
)
 

In [ ]:
#####################

In [23]:

import numpy as np
import pandas as pd
from scipy import stats


def porownaj_rozklady_mack(
    df_reszty,
    ldf,
    sigma2,
    max_okres=8,
    n_resamples_ad=100000,
    seed=12345
):
    """
    Porównanie rozkładów warunkowych modelu Macka:
    - normalny
    - gamma
    - lognormalny

    Zwraca:
    1. tabela_ogolna
    2. tabela_okresy
    3. tabela_porownawcza
    4. reszty_wszystkie

    Wszystkie testy dotyczą reszt PIT
    względem ustalonego N(0,1).

    p-value AD liczone metodą Monte Carlo.
    """

    ldf = np.asarray(ldf, dtype=np.float64)
    sigma2 = np.asarray(sigma2, dtype=np.float64)

    dane = df_reszty.loc[
        (df_reszty["Okres"] >= 1) &
        (df_reszty["Okres"] <= max_okres),
        ["Indeks roku", "Okres", "C_ij", "C_ij1"]
    ].copy()

    dane = dane.replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    okres = dane["Okres"].to_numpy(dtype=int)

    if len(dane) == 0:
        raise ValueError("Brak obserwacji.")

    if okres.max() > len(ldf) or okres.max() > len(sigma2):
        raise ValueError("Brak parametrów dla części okresów.")

    c0 = dane["C_ij"].to_numpy(dtype=float)
    c1 = dane["C_ij1"].to_numpy(dtype=float)

    mu = c0 * ldf[okres - 1]
    var = c0 * sigma2[okres - 1]

    if np.any(mu <= 0) or np.any(var <= 0):
        raise ValueError(
            "Wartości oczekiwane i wariancje muszą być dodatnie."
        )

    # ==========================================
    # 1. DYSTRYBUANTY TRZECH ROZKŁADÓW
    # ==========================================

    # NORMALNY
    u_normal = stats.norm.cdf(
        c1,
        loc=mu,
        scale=np.sqrt(var)
    )

    # GAMMA
    shape_gamma = mu**2 / var
    scale_gamma = var / mu

    u_gamma = stats.gamma.cdf(
        c1,
        a=shape_gamma,
        scale=scale_gamma
    )

    # LOGNORMALNY
    sigma_log2 = np.log1p(var / mu**2)
    sigma_log = np.sqrt(sigma_log2)
    mu_log = np.log(mu) - sigma_log2 / 2

    u_lognormal = stats.lognorm.cdf(
        c1,
        s=sigma_log,
        scale=np.exp(mu_log)
    )


    eps = np.finfo(float).eps

    def to_reszty(u):
        u = np.clip(u, eps, 1.0 - eps)
        return stats.norm.ppf(u)

    reszty = {
        "normal": to_reszty(u_normal),
        "gamma": to_reszty(u_gamma),
        "lognormal": to_reszty(u_lognormal)
    }

    reszty_wszystkie = dane[
        ["Indeks roku", "Okres"]
    ].copy()

    for nazwa, z in reszty.items():
        reszty_wszystkie[nazwa] = z


    def ad_statistic(z, axis=-1):

        z = np.sort(
            np.asarray(z, dtype=np.float64),
            axis=axis
        )

        n = z.shape[axis]

        log_f = stats.norm.logcdf(z)
        log_sf = stats.norm.logsf(
            np.flip(z, axis=axis)
        )

        k = np.arange(1, n + 1)

        shape = [1] * z.ndim
        shape[axis] = n

        weights = (2 * k - 1).reshape(shape)

        return -n - np.sum(
            weights * (log_f + log_sf),
            axis=axis
        ) / n

    def testy(z, rng):

        z = np.asarray(z, dtype=float)

        ks = stats.kstest(
            z,
            "norm",
            args=(0, 1),
            method="auto"
        )

        ad = float(ad_statistic(z))

        if n_resamples_ad is not None:

            wynik_mc = stats.monte_carlo_test(
                data=z,
                rvs=lambda size: rng.standard_normal(size),
                statistic=ad_statistic,
                alternative="greater",
                n_resamples=n_resamples_ad,
                vectorized=True,
                batch=10000
            )

            p_ad = wynik_mc.pvalue

        else:
            p_ad = np.nan

        return {
            "Liczebność próby": len(z),
            "Średnia": np.mean(z),
            "Odchylenie standardowe": np.std(z, ddof=1),
            "Statystyka testowa KS": ks.statistic,
            "p-value KS": ks.pvalue,
            "Statystyka testowa AD": ad,
            "p-value AD": p_ad
        }

    wyniki_globalne = {}
    wiersze_okresy = []

    for nazwa, z in reszty.items():

        rng = np.random.default_rng(seed)

        wyniki_globalne[nazwa] = testy(z, rng)

        for j in range(1, max_okres + 1):

            z_j = z[okres == j]

            if len(z_j) < 2:
                continue

            wynik = testy(z_j, rng)

            wiersze_okresy.append({
                "Rozkład": nazwa,
                "Rok rozwoju": j,
                **wynik
            })

    tabela_ogolna = pd.DataFrame(
        wyniki_globalne
    )

    tabela_ogolna.index.name = "Miara"

    tabela_ogolna = tabela_ogolna.reset_index()

    # ==========================================
    # 7. TABELA WG OKRESÓW
    # ==========================================

    tabela_okresy = pd.DataFrame(
        wiersze_okresy
    )

    tabela_okresy = tabela_okresy.sort_values(
        ["Rok rozwoju", "Rozkład"]
    ).reset_index(drop=True)

    # ==========================================
    # 8. TABELA PORÓWNAWCZA
    # ==========================================

    tabela_porownawcza = pd.DataFrame([
        {
            "Rozkład": nazwa,
            **wyniki_globalne[nazwa]
        }
        for nazwa in ["normal", "gamma", "lognormal"]
    ])

    return (
        tabela_ogolna,
        tabela_okresy,
        tabela_porownawcza,
        reszty_wszystkie
    )


In [24]:
(
    tabela_ogolna,
    tabela_okresy,
    tabela_porownawcza,
    reszty_wszystkie
) = porownaj_rozklady_mack(
    df_reszty=df_reszty,
    ldf=LDF_zaklad,
    sigma2=sigma2_fit,
    max_okres=8,
    n_resamples_ad=100000,
    seed=12345
)

display(tabela_ogolna)
display(tabela_okresy)
display(tabela_porownawcza)

,Miara,normal,gamma,lognormal
0,Liczebność próby,101.000000,101.000000,101.000000
1,Średnia,0.065409,0.065571,0.065652
2,Odchylenie standardowe,1.010779,1.010286,1.010126
3,Statystyka testowa KS,0.078800,0.079407,0.079520
4,p-value KS,0.531313,0.521492,0.519676
5,Statystyka testowa AD,0.988280,0.988676,0.989629
6,p-value AD,0.364076,0.363946,0.363576


,Rozkład,Rok rozwoju,Liczebność próby,Średnia,Odchylenie standardowe,Statystyka testowa KS,p-value KS,Statystyka testowa AD,p-value AD
0,gamma,1,17,0.006853,1.003056,0.136367,0.868818,0.323493,0.919811
1,lognormal,1,17,0.007229,1.004983,0.138454,0.857223,0.328845,0.914961
2,normal,1,17,0.006072,0.999980,0.132249,0.890380,0.314084,0.927711
3,gamma,2,16,0.006390,0.999101,0.146600,0.834033,0.398432,0.849062
4,lognormal,2,16,0.006516,0.998732,0.146586,0.834115,0.397824,0.849542
5,normal,2,16,0.006144,0.999980,0.146658,0.833694,0.399913,0.847512
6,gamma,3,15,0.015762,0.997848,0.208627,0.469124,0.860875,0.435366
7,lognormal,3,15,0.015861,0.996883,0.208164,0.471904,0.857266,0.437626
8,normal,3,15,0.015573,0.999870,0.209563,0.463534,0.868308,0.430296
9,gamma,4,14,-0.099618,1.002767,0.178617,0.699327,0.536405,0.707633


,Rozkład,Liczebność próby,Średnia,Odchylenie standardowe,Statystyka testowa KS,p-value KS,Statystyka testowa AD,p-value AD
0,normal,101,0.065409,1.010779,0.078800,0.531313,0.988280,0.364076
1,gamma,101,0.065571,1.010286,0.079407,0.521492,0.988676,0.363946
2,lognormal,101,0.065652,1.010126,0.079520,0.519676,0.989629,0.363576


In [25]:
for rozklad in ["lognormal"]:

    print("\nROZKŁAD:", rozklad.upper())

    print("\nCAŁA PRÓBA:")
    display(
        tabela_ogolna[["Miara", rozklad]]
    )

    print("\nOKRESY ROZWOJU:")
    display(
        tabela_okresy.loc[
            tabela_okresy["Rozkład"] == rozklad
        ].drop(columns="Rozkład")
    )


ROZKŁAD: LOGNORMAL

CAŁA PRÓBA:


,Miara,lognormal
0,Liczebność próby,101.000000
1,Średnia,0.065652
2,Odchylenie standardowe,1.010126
3,Statystyka testowa KS,0.079520
4,p-value KS,0.519676
5,Statystyka testowa AD,0.989629
6,p-value AD,0.363576



OKRESY ROZWOJU:


,Rok rozwoju,Liczebność próby,Średnia,Odchylenie standardowe,Statystyka testowa KS,p-value KS,Statystyka testowa AD,p-value AD
1,1,17,0.007229,1.004983,0.138454,0.857223,0.328845,0.914961
4,2,16,0.006516,0.998732,0.146586,0.834115,0.397824,0.849542
7,3,15,0.015861,0.996883,0.208164,0.471904,0.857266,0.437626
10,4,14,-0.099575,1.002877,0.178574,0.699599,0.536303,0.707763
13,5,12,0.208616,1.005425,0.233361,0.461664,0.554063,0.689203
16,6,10,0.665147,0.741873,0.447562,0.024045,3.198080,0.022480
19,7,9,0.230266,1.498046,0.194358,0.824746,0.912072,0.406296
22,8,8,-0.458431,0.663967,0.370918,0.170382,1.293794,0.235178
